# Enrichissement leads — Téléphone & site web via Google Places

Ce notebook récupère les 42 leads depuis Supabase, cherche chaque établissement sur Google Places (Text Search + Place Details), et met à jour les colonnes `telephone` et `site_web` dans Supabase.

## 1. Configuration

In [ ]:
# ─── Google Places API ───
GOOGLE_PLACES_API_KEY = ""  # À remplir

# ─── Supabase ───
SUPABASE_URL = "https://hxjryfaakdpwfgseirik.supabase.co"
SUPABASE_API_KEY = "sb_publishable_a7xpn8srwByQrW1rXwpdlw_eQnwAoHT"
SUPABASE_TABLE = "leads"

In [ ]:
import requests
import time

## 2. Chargement des leads depuis Supabase

In [ ]:
def fetch_leads():
    resp = requests.get(
        f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE}?select=siren,nom,ville,code_postal",
        headers={
            "apikey": SUPABASE_API_KEY,
            "Authorization": f"Bearer {SUPABASE_API_KEY}",
            "Accept": "application/json",
        },
    )
    resp.raise_for_status()
    leads = resp.json()
    print(f"✓ {len(leads)} leads chargés depuis Supabase")
    return leads

leads = fetch_leads()

## 3. Recherche Google Places (Text Search → Place Details)

In [ ]:
PLACES_SEARCH_URL = "https://maps.googleapis.com/maps/api/place/textsearch/json"
PLACES_DETAILS_URL = "https://maps.googleapis.com/maps/api/place/details/json"


def search_place(nom, ville):
    """Text Search pour trouver le place_id."""
    resp = requests.get(PLACES_SEARCH_URL, params={
        "query": f"{nom} {ville}",
        "key": GOOGLE_PLACES_API_KEY,
        "language": "fr",
    })
    data = resp.json()
    results = data.get("results", [])
    if not results:
        return None
    return results[0].get("place_id")


def get_place_details(place_id):
    """Place Details pour récupérer téléphone et site web."""
    resp = requests.get(PLACES_DETAILS_URL, params={
        "place_id": place_id,
        "fields": "international_phone_number,website",
        "key": GOOGLE_PLACES_API_KEY,
        "language": "fr",
    })
    result = resp.json().get("result", {})
    return {
        "telephone": result.get("international_phone_number", ""),
        "site_web": result.get("website", ""),
    }


enriched = []
found_phone = 0
found_web = 0

for i, lead in enumerate(leads):
    nom = lead["nom"]
    ville = lead["ville"]
    siren = lead["siren"]
    print(f"[{i+1}/{len(leads)}] {nom[:55]:55s} ", end="")

    place_id = search_place(nom, ville)
    if not place_id:
        print("— ✗ non trouvé sur Google Places")
        enriched.append({"siren": siren, "telephone": "", "site_web": ""})
        time.sleep(0.3)
        continue

    details = get_place_details(place_id)
    tel = details["telephone"]
    web = details["site_web"]

    status_parts = []
    if tel:
        status_parts.append(f"📞 {tel}")
        found_phone += 1
    if web:
        status_parts.append(f"🌐 {web[:40]}")
        found_web += 1
    if status_parts:
        print(f"— ✓ {' | '.join(status_parts)}")
    else:
        print("— ✓ trouvé mais pas de tel/web")

    enriched.append({"siren": siren, "telephone": tel, "site_web": web})
    time.sleep(0.3)

print(f"\n══ Résultat : {found_phone}/{len(leads)} avec téléphone, {found_web}/{len(leads)} avec site web ══")

## 4. Mise à jour Supabase

In [ ]:
updated = 0
skipped = 0

for entry in enriched:
    if not entry["telephone"] and not entry["site_web"]:
        skipped += 1
        continue

    patch_data = {}
    if entry["telephone"]:
        patch_data["telephone"] = entry["telephone"]
    if entry["site_web"]:
        patch_data["site_web"] = entry["site_web"]

    resp = requests.patch(
        f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE}?siren=eq.{entry['siren']}",
        headers={
            "apikey": SUPABASE_API_KEY,
            "Authorization": f"Bearer {SUPABASE_API_KEY}",
            "Content-Type": "application/json",
            "Prefer": "return=minimal",
        },
        json=patch_data,
    )

    if resp.status_code in (200, 204):
        updated += 1
    else:
        print(f"  ✗ SIREN {entry['siren']} : HTTP {resp.status_code} — {resp.text[:200]}")

print(f"\n══ {updated} leads mis à jour, {skipped} sans données à écrire ══")